# Final Project Draft: Breast Cancer Diagnosis Classification

This notebook builds a draft machine learning workflow to predict whether a breast tumor is malignant or benign using the Breast Cancer Wisconsin Diagnostic dataset. The draft includes data loading, exploratory data analysis, engineered features, preprocessing, two classification models, model comparison, visualizations, and reflection on next steps.

## 1. Load libraries and dataset

For this draft, I am using the Breast Cancer Wisconsin Diagnostic dataset available through `scikit-learn`, which is derived from the UCI Machine Learning Repository. This gives me a clean, well-documented classification problem that is appropriate for comparing multiple supervised learning techniques.

In [ ]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

In [ ]:
# Load the dataset from the repository data folder
data_path = 'data/breast_cancer_wisconsin.csv'
df = pd.read_csv(data_path)

target_map = {0: 'malignant', 1: 'benign'}
df['diagnosis_label'] = df['target'].map(target_map)

print(f"Loaded dataset from: {data_path}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 2. Basic statistics and exploratory analysis

This section checks the number of observations, feature types, class balance, and missing values before modeling. Understanding these patterns helps determine preprocessing choices and whether stratification or imputation will be necessary.

In [ ]:
print('Data types:')
display(df.dtypes.to_frame(name='dtype').T)

summary_stats = df.describe().T
summary_stats[['mean', 'std', 'min', 'max']].head(10)

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame(name='missing_values')
missing_summary['missing_pct'] = (missing_summary['missing_values'] / len(df) * 100).round(2)
display(missing_summary[missing_summary['missing_values'] > 0])

if missing_summary['missing_values'].sum() == 0:
    print('No missing values were found in this dataset.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='diagnosis_label', palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Target Distribution')
axes[0, 0].set_xlabel('Diagnosis')
axes[0, 0].set_ylabel('Count')

sns.boxplot(data=df, x='diagnosis_label', y='mean radius', palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Mean Radius by Diagnosis')
axes[0, 1].set_xlabel('Diagnosis')
axes[0, 1].set_ylabel('Mean Radius')

sns.boxplot(data=df, x='diagnosis_label', y='mean texture', palette='Set2', ax=axes[1, 0])
axes[1, 0].set_title('Mean Texture by Diagnosis')
axes[1, 0].set_xlabel('Diagnosis')
axes[1, 0].set_ylabel('Mean Texture')

sns.scatterplot(
    data=df,
    x='mean perimeter',
    y='mean area',
    hue='diagnosis_label',
    palette='Set1',
    alpha=0.8,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Mean Area vs Mean Perimeter')
axes[1, 1].set_xlabel('Mean Perimeter')
axes[1, 1].set_ylabel('Mean Area')

plt.tight_layout()
plt.show()

**Key data characteristics:**

The dataset contains 569 observations with numerical predictor variables describing tumor cell nuclei and a binary classification target. The target is moderately imbalanced, with more benign cases than malignant cases, so I will use a stratified train/test split to preserve class proportions. The dataset has no missing values, but I will still include imputation in preprocessing so the workflow remains robust and reusable if I extend the feature set later.

## 3. Engineer custom features

I created three engineered features that capture relationships between tumor size, boundary complexity, and texture. These features are intended to provide signal that may not be as obvious when the raw measurements are considered independently.

In [ ]:
df_fe = df.copy()

# Feature 1: size relative to boundary length
# Larger values suggest more area for a given perimeter.
df_fe['area_perimeter_ratio'] = df_fe['mean area'] / (df_fe['mean perimeter'] + 1e-6)

# Feature 2: interaction between tumor size and texture
# Captures cases where rough texture and larger radius appear together.
df_fe['radius_texture_interaction'] = df_fe['mean radius'] * df_fe['mean texture']

# Feature 3: combined surface smoothness and symmetry signal
# Helps summarize shape regularity in one feature.
df_fe['smoothness_symmetry_product'] = df_fe['mean smoothness'] * df_fe['mean symmetry']

engineered_cols = [
    'area_perimeter_ratio',
    'radius_texture_interaction',
    'smoothness_symmetry_product'
]

display(df_fe[engineered_cols + ['diagnosis_label']].head())

**Engineered feature justifications:**

1. `area_perimeter_ratio`: This feature measures how much area a tumor occupies relative to its perimeter. In domain terms, it can help capture shape compactness, which may differ between benign and malignant masses.

2. `radius_texture_interaction`: This interaction feature combines tumor size and texture intensity. It should help if texture becomes more predictive when the tumor is also physically larger.

3. `smoothness_symmetry_product`: This feature combines two shape-related signals into a single value. It is motivated by the idea that malignant tumors may show different combinations of surface smoothness and symmetry than benign tumors.

## 4. Prepare data for modeling

I will use an 80/20 train/test split with stratification because this is a classification problem with some class imbalance. I am applying median imputation and standard scaling to the numeric features so the preprocessing is consistent across both models, especially because Logistic Regression is sensitive to feature scale.

In [ ]:
feature_cols = [col for col in df_fe.columns if col not in ['target', 'diagnosis_label']]
X = df_fe[feature_cols]
y = df_fe['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train class balance:')
print(y_train.value_counts(normalize=True).sort_index())
print('Test class balance:')
print(y_test.value_counts(normalize=True).sort_index())

In [ ]:
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_processed = scaler.fit_transform(X_train_imputed)
X_test_processed = scaler.transform(X_test_imputed)

X_train_processed = pd.DataFrame(X_train_processed, columns=feature_cols, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_cols, index=X_test.index)

print('Preprocessing complete.')
print('Scaled train matrix shape:', X_train_processed.shape)

## 5. Model 1: Random Forest Classifier

Random Forest is a strong tree-based baseline for classification tasks because it can model nonlinear relationships and interactions without requiring extensive manual transformation.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    random_state=RANDOM_STATE
)

start_time = time.time()
rf.fit(X_train_processed, y_train)
rf_time = time.time() - start_time

y_pred_rf = rf.predict(X_test_processed)
y_prob_rf = rf.predict_proba(X_test_processed)[:, 1]

rf_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf),
    'Recall': recall_score(y_test, y_pred_rf),
    'F1 Score': f1_score(y_test, y_pred_rf),
    'ROC AUC': roc_auc_score(y_test, y_prob_rf),
    'Training Time (s)': rf_time,
}

pd.Series(rf_metrics).round(4)

In [ ]:
rf_cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

print(classification_report(y_test, y_pred_rf, target_names=['malignant', 'benign']))

## 6. Model 2: Logistic Regression

Logistic Regression provides a more interpretable linear baseline. It is useful for comparison because it tends to perform well on structured classification problems when the classes are reasonably separable after scaling.

In [ ]:
lr = LogisticRegression(
    max_iter=2000,
    solver='lbfgs',
    random_state=RANDOM_STATE
)

start_time = time.time()
lr.fit(X_train_processed, y_train)
lr_time = time.time() - start_time

y_pred_lr = lr.predict(X_test_processed)
y_prob_lr = lr.predict_proba(X_test_processed)[:, 1]

lr_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr),
    'Recall': recall_score(y_test, y_pred_lr),
    'F1 Score': f1_score(y_test, y_pred_lr),
    'ROC AUC': roc_auc_score(y_test, y_prob_lr),
    'Training Time (s)': lr_time,
}

pd.Series(lr_metrics).round(4)

In [ ]:
lr_cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6, 5))
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

print(classification_report(y_test, y_pred_lr, target_names=['malignant', 'benign']))

## 7. Compare model performance

Both models were trained on the same processed training data and evaluated using the same test split and classification metrics so the comparison is fair.

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Key Hyperparameters': 'n_estimators=200, max_depth=8, min_samples_split=4',
        'Accuracy': rf_metrics['Accuracy'],
        'Precision': rf_metrics['Precision'],
        'Recall': rf_metrics['Recall'],
        'F1 Score': rf_metrics['F1 Score'],
        'ROC AUC': rf_metrics['ROC AUC'],
        'Training Time (s)': rf_metrics['Training Time (s)'],
        'Interpretability': 'Moderate',
        'Overfitting Risk': 'Medium'
    },
    {
        'Model': 'Logistic Regression',
        'Key Hyperparameters': 'max_iter=2000, solver=lbfgs',
        'Accuracy': lr_metrics['Accuracy'],
        'Precision': lr_metrics['Precision'],
        'Recall': lr_metrics['Recall'],
        'F1 Score': lr_metrics['F1 Score'],
        'ROC AUC': lr_metrics['ROC AUC'],
        'Training Time (s)': lr_metrics['Training Time (s)'],
        'Interpretability': 'High',
        'Overfitting Risk': 'Low to medium'
    }
])

results_rounded = results.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Training Time (s)']:
    results_rounded[col] = results_rounded[col].round(4)

display(results_rounded)

**Model comparison analysis:**

In this draft, Logistic Regression slightly outperformed Random Forest on the held-out test set, with stronger accuracy, F1 score, and ROC AUC while also training faster. That result suggests the classes are fairly well separated with the current features and scaling, so a linear decision boundary is already capturing much of the signal. Random Forest is still valuable because it can model nonlinear interactions automatically, but it comes with higher training cost and somewhat lower interpretability. At this stage, I am leaning toward Logistic Regression for the final submission unless later tuning shows a meaningful gain from the tree-based approach.

## 8. Training and validation visualization

To understand whether the tree-based model is benefiting from more data or showing signs of overfitting, I plotted a learning curve for the Random Forest model.

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=rf,
    X=X_train_processed,
    y=y_train,
    cv=5,
    scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=None
)

train_scores_mean = train_scores.mean(axis=1)
validation_scores_mean = validation_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores_mean, marker='o', label='Training F1')
plt.plot(train_sizes, validation_scores_mean, marker='o', label='Validation F1')
plt.title('Random Forest Learning Curve')
plt.xlabel('Training Examples')
plt.ylabel('F1 Score')
plt.legend()
plt.show()

## 9. Reflection and next steps

**Feature engineering plans:** For the final version, I plan to add at least two more features. Likely additions include a texture-to-perimeter ratio, a compactness-based interaction, and potentially selected polynomial terms for the most predictive geometric measurements. I also want to compare whether feature selection improves performance by reducing redundant measurements.

**Model optimization plans:** The next step is hyperparameter tuning. For Random Forest, I would test values for `max_depth`, `min_samples_leaf`, and `n_estimators`. For Logistic Regression, I would tune the regularization strength and compare penalty settings. I also want to evaluate cross-validation metrics to make the performance estimate more stable than a single train/test split.

**Questions for instructor feedback:** I would like feedback on whether my engineered features are sufficiently domain-informed or whether I should push further into shape-complexity features. I would also appreciate guidance on whether ROC AUC and F1 should remain my primary evaluation metrics, or whether recall should receive more emphasis because false negatives are especially costly in a diagnosis setting.